In [ ]:
import json, csv
from nltk.corpus import wordnet as wn
import networkx as nx

# Build the legit-synset graph
G = nx.DiGraph()
G.add_nodes_from(x.name() for x in wn.all_synsets() if x.pos() == "n")
for child in wn.all_synsets():
    if child.pos() != "n":
        continue

    for parent in child.hypernyms():
        if child.pos() != "n":
            continue
        G.add_edge(parent.name(), child.name())

# Add the illegit-synset graph
with open(r"C:\Users\cgokmen\research\bddl\utils\custom_synsets.csv") as f:
    reader = csv.DictReader(f)
    for row in reader:
        child = row["custom_synset"].strip()
        parent = wn.synset(row["hypernyms"].strip()).name()
        assert parent in G.nodes, "Could not find " + parent
        G.add_edge(parent, child)

In [ ]:
# Get all the category mappings
import collections

categories = []
synset_to_cat_and_mass = collections.defaultdict(dict)
with open(r"D:\BEHAVIOR-1K\asset_pipeline\metadata\category_mapping.csv") as f:
    reader = csv.DictReader(f)
    for row in reader:
        s = row["synset"].strip()
        try:
            s = wn.synset(s).name()
        except:
            pass
        c = row["category"].strip()
        mass_str = row["mass"].strip()
        m = float(mass_str) if mass_str else None
        if s not in G.nodes:
            print("Could not find ", s, c)
            # continue
        synset_to_cat_and_mass[s][c] = m
        categories.append((c, s))

In [ ]:
# Add a list of masses to every node, going from bottom to top
topological_order = list(reversed(list(nx.topological_sort(G))))
for node in topological_order:
    # Get this node's immediate children. Their stuff has already been collected
    this_cat_and_mass = {}
    for succ in G.successors(node):
        this_cat_and_mass.update(G.nodes[succ]["cat_and_mass"])

    # Add the stuff that belongs to this particular node
    this_cat_and_mass.update(synset_to_cat_and_mass[node])
    G.nodes[node]["cat_and_mass"] = this_cat_and_mass

In [ ]:
import numpy as np

# Get the mean mass for each node
for node in G.nodes:
    this_cat_and_mass = G.nodes[node]["cat_and_mass"]
    all_masses = [mass for mass in this_cat_and_mass.values() if mass is not None]
    if all_masses:
        G.nodes[node]["mean_mass"] = np.mean(all_masses) if all_masses else None
        # G.nodes[node]["estimation_distance"] = 0

In [ ]:
# Propagate the masses down now.
reverse_topological_order = list(nx.topological_sort(G))
for node in reverse_topological_order:
    if "mean_mass" in G.nodes[node]:
        continue
    parents = list(G.predecessors(node))
    parent_masses = [
        G.nodes[p]["mean_mass"] for p in parents if "mean_mass" in G.nodes[p]
    ]
    if not parent_masses:
        continue
    G.nodes[node]["mean_mass"] = np.mean(parent_masses)
    # G.nodes[node]["estimation_distance"] = G.nodes[parent]["estimation_distance"] + 1

In [ ]:
for c, s in categories:
    mass = G.nodes[s]["mean_mass"]
    print(mass)

In [ ]:
with open(
    r"D:\BEHAVIOR-1K\asset_pipeline\artifacts\pipeline\object_inventory.json", "r"
) as f:
    inv = json.load(f)

In [ ]:
provided_cats = {x.split("-")[0] for x in inv["providers"].keys()}

In [ ]:
for cat, _ in categories:
    print(int(cat in provided_cats))

In [ ]:
len(categories)